# 05 Model LSTM - Rolling Backtest 24h

## Analysis of `lstm_2019-2025.ipynb` (fit-for-purpose gaps)
1. Uses a generic 80/20 row split, not the business split (`train < 2025-01-01`, test year 2025).
2. Not a rolling day-ahead backtest: it trains once and predicts on a fixed test block.
3. Scoping mismatch for day-ahead operations: no explicit daily `h=24` cycle simulation.
4. Feature scope mismatch: mostly univariate `price`, while your project uses known day-ahead exogenous drivers.
5. Evaluation is mostly visual and flattened, not cycle-based by daily forecast cutoffs.

## Changes in This Notebook
1. Uses same scope as MLForecast: rolling daily backtest with `h=24`.
2. Uses explicit date split: train history from 2019, rolling test over 2025.
3. Keeps key exogenous features including `price_gas`.
4. Avoids leakage by fitting scalers only on each cycle's training history.
5. Uses point-forecast metrics only (RMSE, MAE, MAPE).

## Architecture
Two-input LSTM:
- Input A: past window (lookback) with target + historical exogenous context.
- Input B: known exogenous values for the next 24h (flattened).
- Output: next 24 hourly prices.

In [3]:
# If needed:
# %pip install tensorflow scikit-learn

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

import tensorflow as tf
from tensorflow.keras import Model
from tensorflow.keras.layers import Input, LSTM, Dense, Dropout, Concatenate
from tensorflow.keras.callbacks import EarlyStopping

tf.random.set_seed(42)
np.random.seed(42)


In [4]:
# Config
DATA_PATH = '../../data_cleaned/merged/02_4_clean_data.csv'
TARGET = 'price'
H = 24
LOOKBACK = 24 * 7   # 7 days
MIN_TRAIN_SAMPLES = 1000
EPOCHS = 8
BATCH_SIZE = 64
RETRAIN_EVERY_DAYS = 1  # keep 1 for strict daily refit; increase for speed
MAX_CYCLES = None       # set int for faster debug runs

start_date = pd.Timestamp('2019-01-01 00:00:00')
split_date = pd.Timestamp('2025-01-01 00:00:00')
end_date = pd.Timestamp('2026-01-01 00:00:00')


In [5]:
# Load and align data
df = pd.read_csv(DATA_PATH)
df['period_start_utc'] = pd.to_datetime(df['period_start_utc'], errors='coerce', utc=True).dt.tz_localize(None)
df['date'] = pd.to_datetime(df['date'], errors='coerce')

# Same project range
df = df[(df['date'] >= start_date) & (df['date'] < end_date)].copy()

# Same year-dummy treatment used in the XGBoost/MLForecast scope
df = pd.get_dummies(df, columns=['year'], drop_first=True)

df = df.rename(columns={'period_start_utc': 'ds', TARGET: 'y'})
df = df.sort_values('ds').reset_index(drop=True)

print(f'Rows: {len(df):,}')
print(f'Range: {df.ds.min()} -> {df.ds.max()}')


Rows: 61,367
Range: 2019-01-01 00:00:00 -> 2025-12-31 22:00:00


In [6]:
# Features
exog_base = [
    'load_forecast_da', 'res_sum_da', 'gen_forecast_da',
    'dayofyear_sin1', 'dayofyear_cos1',
    'hour_sin1', 'hour_cos1',
    'dayofweek_sin1', 'dayofweek_cos1',
    'is_holiday', 'price_gas'
]
year_dummies = [c for c in df.columns if c.startswith('year_')]
exog_cols = exog_base + year_dummies
past_cols = ['y'] + exog_cols

missing = [c for c in past_cols + ['ds'] if c not in df.columns]
if missing:
    raise ValueError(f'Missing required columns: {missing}')

model_df = df[['ds', 'y'] + exog_cols].copy()
print(f'Exogenous features: {len(exog_cols)}')


Exogenous features: 17


In [7]:
# Build rolling daily cutoffs over test period (2025)
cutoffs = pd.date_range(start=split_date, end=end_date - pd.Timedelta(days=1), freq='D')
if MAX_CYCLES is not None:
    cutoffs = cutoffs[:MAX_CYCLES]
print(f'Forecast cycles: {len(cutoffs)}')


Forecast cycles: 365


In [8]:
# Helpers
def build_supervised(train_hist, lookback, h, past_cols, exog_cols):
    X_past, X_fut, Y = [], [], []
    n = len(train_hist)
    for t in range(lookback, n - h + 1):
        past_block = train_hist.iloc[t - lookback:t][past_cols].values
        fut_exog = train_hist.iloc[t:t + h][exog_cols].values.reshape(-1)
        target = train_hist.iloc[t:t + h]['y'].values
        X_past.append(past_block)
        X_fut.append(fut_exog)
        Y.append(target)
    return np.array(X_past), np.array(X_fut), np.array(Y)

def build_model(lookback, n_past_features, h, n_exog):
    inp_past = Input(shape=(lookback, n_past_features), name='past_seq')
    x = LSTM(64, return_sequences=False)(inp_past)
    x = Dropout(0.2)(x)
    x = Dense(32, activation='relu')(x)

    inp_fut = Input(shape=(h * n_exog,), name='future_exog')
    z = Dense(64, activation='relu')(inp_fut)
    z = Dropout(0.1)(z)

    c = Concatenate()([x, z])
    c = Dense(64, activation='relu')(c)
    out = Dense(h, name='y_hat')(c)

    model = Model(inputs=[inp_past, inp_fut], outputs=out)
    model.compile(optimizer='adam', loss='mse')
    return model


In [9]:
# Rolling daily backtest
predictions = []
model = None

for i, cutoff in enumerate(cutoffs, 1):
    train_hist = model_df[model_df['ds'] < cutoff].copy()
    horizon_end = cutoff + pd.Timedelta(hours=H)
    test_h = model_df[(model_df['ds'] >= cutoff) & (model_df['ds'] < horizon_end)].copy()

    if len(test_h) != H:
        continue

    # Fit scalers on train history only (no test leakage)
    past_scaler = StandardScaler()
    exog_scaler = StandardScaler()
    y_scaler = StandardScaler()

    train_hist_scaled = train_hist.copy()
    train_hist_scaled[past_cols] = past_scaler.fit_transform(train_hist[past_cols])
    train_hist_scaled[exog_cols] = exog_scaler.fit_transform(train_hist[exog_cols])
    train_hist_scaled['y'] = y_scaler.fit_transform(train_hist[['y']])

    X_past, X_fut, Y = build_supervised(train_hist_scaled, LOOKBACK, H, past_cols, exog_cols)
    if len(X_past) < MIN_TRAIN_SAMPLES:
        continue

    if (i - 1) % RETRAIN_EVERY_DAYS == 0 or model is None:
        model = build_model(LOOKBACK, len(past_cols), H, len(exog_cols))
        es = EarlyStopping(monitor='val_loss', patience=2, restore_best_weights=True)
        model.fit([X_past, X_fut], Y, epochs=EPOCHS, batch_size=BATCH_SIZE, validation_split=0.1, callbacks=[es], verbose=0)

    # Build one-step day-ahead input at this cutoff
    past_block = train_hist.tail(LOOKBACK).copy()
    if len(past_block) < LOOKBACK:
        continue

    past_scaled = past_scaler.transform(past_block[past_cols]).reshape(1, LOOKBACK, len(past_cols))
    fut_scaled = exog_scaler.transform(test_h[exog_cols]).reshape(1, H * len(exog_cols))

    pred_scaled = model.predict([past_scaled, fut_scaled], verbose=0).reshape(-1, 1)
    pred = y_scaler.inverse_transform(pred_scaled).reshape(-1)

    out = test_h[['ds', 'y']].copy()
    out['prediction'] = pred
    out['cutoff'] = cutoff
    predictions.append(out)

    if i % 15 == 0:
        print(f'Processed {i}/{len(cutoffs)} cycles')

results = pd.concat(predictions, ignore_index=True).rename(columns={'y': 'actual'})
results.head()


KeyboardInterrupt: 

In [ ]:
# Metrics (point forecast only)
rmse = mean_squared_error(results['actual'], results['prediction'], squared=False)
mae = mean_absolute_error(results['actual'], results['prediction'])
eps = 1e-8
mape = np.mean(np.abs((results['actual'] - results['prediction']) / np.maximum(np.abs(results['actual']), eps))) * 100

print(f'RMSE: {rmse:,.4f}')
print(f'MAE:  {mae:,.4f}')
print(f'MAPE: {mape:,.2f}%')


In [ ]:
# Plot full test period
plot_df = results.set_index('ds').sort_index()
ax = plot_df[['actual']].plot(figsize=(15, 5), title='LSTM Rolling Daily Backtest (h=24)')
plot_df['prediction'].plot(ax=ax, alpha=0.85)
ax.legend(['actual', 'prediction'])
plt.show()


In [ ]:
# Zoom into one week
zoom_start = split_date + pd.Timedelta(days=90)
zoom_end = zoom_start + pd.Timedelta(days=7)
zoom = plot_df[(plot_df.index >= zoom_start) & (plot_df.index < zoom_end)]

ax = zoom[['actual']].plot(figsize=(15, 5), title='Zoom: 7-day window')
zoom['prediction'].plot(ax=ax, style='.')
ax.legend(['actual', 'prediction'])
plt.show()


## Notes
- This is stricter and operationally closer than a one-shot split, but computationally heavier.
- If runtime is too long, first set `MAX_CYCLES=30` for a quick benchmark, then restore full-year backtest.
- You can also set `RETRAIN_EVERY_DAYS=2` or `3` to reduce retraining cost.

In [ ]:
# Unified metrics + export for comparison
from pathlib import Path

# normalize evaluation frame name
if 'results' in locals():
    eval_df = results.copy()
elif 'res' in locals():
    eval_df = res.copy()
else:
    raise ValueError('No results dataframe found (expected `results` or `res`).')

# normalize column names
if 'y' in eval_df.columns and 'actual' not in eval_df.columns:
    eval_df = eval_df.rename(columns={'y': 'actual'})
if 'xgb' in eval_df.columns and 'prediction' not in eval_df.columns:
    eval_df = eval_df.rename(columns={'xgb': 'prediction'})
if 'ds' not in eval_df.columns and eval_df.index.name is not None:
    eval_df = eval_df.reset_index()

required_cols = {'actual', 'prediction'}
missing = required_cols - set(eval_df.columns)
if missing:
    raise ValueError(f'Missing required columns for metrics/export: {missing}')

# robust metrics for power prices (can be near zero/negative)
rmse = mean_squared_error(eval_df['actual'], eval_df['prediction'], squared=False)
mae = mean_absolute_error(eval_df['actual'], eval_df['prediction'])
smape = 100 * np.mean(
    2 * np.abs(eval_df['actual'] - eval_df['prediction']) /
    (np.abs(eval_df['actual']) + np.abs(eval_df['prediction']) + 1e-8)
)

mask = np.abs(eval_df['actual']) >= 10
mape_filtered = (
    np.mean(
        np.abs((eval_df.loc[mask, 'actual'] - eval_df.loc[mask, 'prediction']) /
               np.abs(eval_df.loc[mask, 'actual']))
    ) * 100
    if mask.any() else np.nan
)

print(f'RMSE: {rmse:,.4f}')
print(f'MAE:  {mae:,.4f}')
print(f'sMAPE: {smape:,.2f}%')
print(f'MAPE (|actual|>=10): {mape_filtered:,.2f}%')

out_dir = Path('../../artifacts/model_results')
out_dir.mkdir(parents=True, exist_ok=True)

metrics_df = pd.DataFrame([{
    'model': 'LSTM',
    'rmse': rmse,
    'mae': mae,
    'smape': smape,
    'mape_filtered_abs_ge_10': mape_filtered,
    'n_predictions': len(eval_df)
}])

metrics_df.to_csv(out_dir / 'lstm_metrics_rolling_24h.csv', index=False)

pred_cols = [c for c in ['unique_id', 'cutoff', 'ds', 'actual', 'prediction'] if c in eval_df.columns]
eval_df[pred_cols].to_csv(out_dir / 'lstm_predictions_rolling_24h.csv', index=False)

metrics_df
